# 01 — MODFLOW 6 Baseline Run Check

**Purpose:** Run the copied Little Plover River MODFLOW 6 steady-state model from this repository.

This notebook checks that the model can be:

1. Found in the clean source model folder.
2. Copied into a safe `results/` run folder.
3. Loaded with FloPy.
4. Run with the `mf6` executable.
5. Checked for successful completion.
6. Summarized using the generated output files.

## Important idea

We will **not** run the model directly inside:

```text
models/lpr_mf6/steady_state/
```

That folder is treated as the clean, original source model copied from `LPR_redux`.

Instead, this notebook copies the model into:

```text
results/mf6_runs/baseline_steady_state/
```

and runs MODFLOW there. This keeps the original model files clean.


## 1. Imports and project-root setup

In [29]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
from datetime import datetime
import pandas as pd

def find_project_root(start=None):
    """Find the repository root by looking for common project marker files."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "environment.yml").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Try opening Jupyter from the Modeling-Uncertainties repo root."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")
print(f"Python executable: {sys.executable}")


Project root: /workspaces/Modeling-Uncertainties
Current working directory: /workspaces/Modeling-Uncertainties
Python executable: /opt/conda/envs/gw_uncertainty/bin/python


## 2. Define source and run paths

In [30]:
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MF6_RESULTS_DIR = RESULTS_DIR / "mf6_runs"

# Clean source model copied from LPR_redux
LPR_MF6_SOURCE_DIR = MODELS_DIR / "lpr_mf6" / "steady_state"

# Safe run workspace for this notebook
BASELINE_RUN_DIR = MF6_RESULTS_DIR / "baseline_steady_state"

print(f"Source model folder: {LPR_MF6_SOURCE_DIR}")
print(f"Baseline run folder: {BASELINE_RUN_DIR}")


Source model folder: /workspaces/Modeling-Uncertainties/models/lpr_mf6/steady_state
Baseline run folder: /workspaces/Modeling-Uncertainties/results/mf6_runs/baseline_steady_state


## 3. Check the source MODFLOW 6 model files

This checks the clean source folder before copying it into the run folder.


In [31]:
required_mf6_files = [
    "mfsim.nam",
    "lpr_ss.tdis",
    "lpr_ss.ims",
    "lpr_ss_gwf.nam",
    "lpr_ss_gwf.dis",
    "lpr_ss_gwf.npf",
    "lpr_ss_gwf.ic",
    "lpr_ss_gwf.oc",
    "lpr_ss_gwf.rcha",
    "lpr_ss_gwf.wel",
    "lpr_ss_gwf.drn",
    "lpr_ss_gwf.chd",
    "lpr_ss.sfr",
    "sfr_obs.obs",
]

source_file_check = pd.DataFrame(
    [
        {
            "file": filename,
            "relative_path": str((LPR_MF6_SOURCE_DIR / filename).relative_to(PROJECT_ROOT)),
            "exists": (LPR_MF6_SOURCE_DIR / filename).exists(),
            "size_MB": round((LPR_MF6_SOURCE_DIR / filename).stat().st_size / 1_000_000, 3)
            if (LPR_MF6_SOURCE_DIR / filename).exists()
            else None,
        }
        for filename in required_mf6_files
    ]
)

source_file_check


,file,relative_path,exists,size_MB
0,mfsim.nam,models/lpr_mf6/steady_state/mfsim.nam,True,0.000
1,lpr_ss.tdis,models/lpr_mf6/steady_state/lpr_ss.tdis,True,0.000
2,lpr_ss.ims,models/lpr_mf6/steady_state/lpr_ss.ims,True,0.001
3,lpr_ss_gwf.nam,models/lpr_mf6/steady_state/lpr_ss_gwf.nam,True,0.000
4,lpr_ss_gwf.dis,models/lpr_mf6/steady_state/lpr_ss_gwf.dis,True,0.001
5,lpr_ss_gwf.npf,models/lpr_mf6/steady_state/lpr_ss_gwf.npf,True,0.001
6,lpr_ss_gwf.ic,models/lpr_mf6/steady_state/lpr_ss_gwf.ic,True,0.000
7,lpr_ss_gwf.oc,models/lpr_mf6/steady_state/lpr_ss_gwf.oc,True,0.000
8,lpr_ss_gwf.rcha,models/lpr_mf6/steady_state/lpr_ss_gwf.rcha,True,0.000
9,lpr_ss_gwf.wel,models/lpr_mf6/steady_state/lpr_ss_gwf.wel,True,0.000


In [32]:
missing_files = source_file_check.loc[~source_file_check["exists"], "file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing required MODFLOW 6 source files: {missing_files}")

print("All required MODFLOW 6 source files were found.")
print(f"Total files in source model folder: {len(list(LPR_MF6_SOURCE_DIR.glob('*')))}")


All required MODFLOW 6 source files were found.
Total files in source model folder: 38


## 4. Copy the source model into a safe run folder

Set `OVERWRITE_RUN_FOLDER = True` to make this notebook repeatable.

This will delete and recreate only:

```text
results/mf6_runs/baseline_steady_state/
```

It will **not** modify the clean source folder.


In [33]:
OVERWRITE_RUN_FOLDER = True

if BASELINE_RUN_DIR.exists():
    if OVERWRITE_RUN_FOLDER:
        print(f"Removing existing run folder: {BASELINE_RUN_DIR.relative_to(PROJECT_ROOT)}")
        shutil.rmtree(BASELINE_RUN_DIR)
    else:
        raise FileExistsError(
            f"The run folder already exists: {BASELINE_RUN_DIR}. "
            "Set OVERWRITE_RUN_FOLDER = True if you want to replace it."
        )

BASELINE_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LPR_MF6_SOURCE_DIR, BASELINE_RUN_DIR)

print(f"Copied source model to: {BASELINE_RUN_DIR.relative_to(PROJECT_ROOT)}")
print(f"Total files in run folder: {len(list(BASELINE_RUN_DIR.glob('*')))}")


Removing existing run folder: results/mf6_runs/baseline_steady_state


Copied source model to: results/mf6_runs/baseline_steady_state
Total files in run folder: 38


## 5. Inspect the copied run folder

In [34]:
run_files = sorted([p for p in BASELINE_RUN_DIR.iterdir() if p.is_file()])

run_file_table = pd.DataFrame(
    [
        {
            "file": p.name,
            "size_MB": round(p.stat().st_size / 1_000_000, 3),
        }
        for p in run_files
    ]
)

run_file_table.head(20)


,file,size_MB
0,lpr_ss.ims,0.001
1,lpr_ss.sfr,0.400
2,lpr_ss.sfr.orig,0.400
3,lpr_ss.tdis,0.000
4,lpr_ss_gwf.chd,0.000
5,lpr_ss_gwf.chd_stress_period_data_1.txt,0.141
6,lpr_ss_gwf.dis,0.001
7,lpr_ss_gwf.dis_botm_layer1.txt,16.420
8,lpr_ss_gwf.dis_botm_layer2.txt,16.420
9,lpr_ss_gwf.dis_botm_layer3.txt,16.420


## 6. Load the copied model with FloPy

This does not run MODFLOW yet. It only checks that FloPy can read the copied input files.


In [35]:
import flopy

sim = flopy.mf6.MFSimulation.load(
    sim_ws=str(BASELINE_RUN_DIR),
    verbosity_level=1,
)

print("MODFLOW 6 simulation loaded successfully with FloPy.")
print(f"Simulation name: {sim.name}")
print(f"Model names: {sim.model_names}")

gwf = sim.get_model()
print(f"Groundwater model name: {gwf.name}")

print("\nPackages:")
for package in gwf.packagelist:
    print(f"  - {package.package_type:8s} {package.filename}")


loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package sfr...
    loading package ic...
    loading package npf...
    loading package oc...
    loading package rch...
    loading package chd...
    loading package drn...
    loading package wel...
  loading solution package lpr_ss_gwf...
MODFLOW 6 simulation loaded successfully with FloPy.
Simulation name: modflowsim
Model names: ['lpr_ss_gwf']
Groundwater model name: lpr_ss_gwf

Packages:
  - dis      lpr_ss_gwf.dis
  - obs      sfr_obs.obs
  - sfr      lpr_ss.sfr
  - ic       lpr_ss_gwf.ic
  - npf      lpr_ss_gwf.npf
  - oc       lpr_ss_gwf.oc
  - rcha     lpr_ss_gwf.rcha
  - chd      lpr_ss_gwf.chd
  - drn      lpr_ss_gwf.drn
  - wel      lpr_ss_gwf.wel


## 7. Basic model information from FloPy

In [36]:
# Discretization information
dis = gwf.get_package("dis")

nlay = int(dis.nlay.get_data())
nrow = int(dis.nrow.get_data())
ncol = int(dis.ncol.get_data())

print(f"Number of layers: {nlay}")
print(f"Number of rows:   {nrow}")
print(f"Number of cols:   {ncol}")
print(f"Total cells:      {nlay * nrow * ncol:,}")

# Stress period information from TDIS
tdis = sim.get_package("tdis")
perioddata = tdis.perioddata.get_data()

period_table = pd.DataFrame(
    perioddata,
    columns=["perlen", "nstp", "tsmult"]
)

period_table


Number of layers: 3
Number of rows:   900
Number of cols:   1070
Total cells:      2,889,000


,perlen,nstp,tsmult
0,1.0,1,1.0


## 8. Check the `mf6` executable

Before running the model, make sure the `mf6` executable is available from the active environment.


In [37]:
mf6_path = shutil.which("mf6")

if mf6_path is None:
    raise FileNotFoundError(
        "The mf6 executable was not found. Activate your project environment with: conda activate gw_uncertainty"
    )

print(f"mf6 executable found at: {mf6_path}")

version_result = subprocess.run(
    ["mf6", "-v"],
    capture_output=True,
    text=True,
    timeout=30,
)

print("mf6 version output:")
print(version_result.stdout.strip() or version_result.stderr.strip())


mf6 executable found at: /opt/conda/envs/gw_uncertainty/bin/mf6
mf6 version output:
mf6: 6.6.3 09/29/2025


## 9. Run MODFLOW 6 in the copied run folder

This is the first real model run from the clean repository workflow.

The model will run inside:

```text
results/mf6_runs/baseline_steady_state/
```


In [38]:
start_time = datetime.now()

success, buff = sim.run_simulation(
    silent=False,
    report=True,
)

end_time = datetime.now()
elapsed = end_time - start_time

run_stdout_path = BASELINE_RUN_DIR / "run_stdout_flopy.txt"
run_stdout_path.write_text("\n".join(buff), encoding="utf-8")

print(f"Run success: {success}")
print(f"Elapsed time: {elapsed}")
print(f"Saved run report to: {run_stdout_path.relative_to(PROJECT_ROOT)}")

FloPy is using the following executable to run the model: ../../../../../opt/conda/envs/gw_uncertainty/bin/mf6
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.6.3 09/29/2025

   MODFLOW 6 compiled Oct 07 2025 22:51:46 with Intel(R) Fortran Intel(R) 64
   Compiler Classic for applications running on Intel(R) 64, Version 2021.7.0
                             Build 20220726_000000

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warranty, expressed or 
implied, is made by the USGS or the U.S. Government as to the 
functionality of the software and related material nor shall the 
fact of release constitute any such warranty. Furthermore, the 
software is released on condition that neither the 

    Solving:  Stress period:     1    Time step:     1
 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/04/19  5:00:14
 Elapsed run time:  8 Minutes, 50.401 Seconds
 

WARNING REPORT:

  1. OPTIONS BLOCK VARIABLE 'UNIT_CONVERSION' IN FILE 'lpr_ss.sfr' WAS
     DEPRECATED IN VERSION 6.4.2. SETTING UNIT_CONVERSION DIRECTLY.
  2. SFR Package SFR_0 has 1 reach(es) with zero connections.
 Normal termination of simulation.
Run success: True
Elapsed time: 0:08:50.476507
Saved run report to: results/mf6_runs/baseline_steady_state/run_stdout_flopy.txt


## 10. Check whether the MODFLOW run succeeded

In [39]:
buff_text = "\n".join(buff).lower()

normal_termination = "normal termination" in buff_text
run_success = bool(success) and normal_termination

run_summary = pd.DataFrame(
    [
        {"check": "FloPy reported success", "passed": bool(success)},
        {"check": "MODFLOW reported normal termination", "passed": normal_termination},
        {"check": "Overall run success", "passed": run_success},
    ]
)

run_summary

,check,passed
0,FloPy reported success,True
1,MODFLOW reported normal termination,True
2,Overall run success,True


In [40]:
if not run_success:
    raise RuntimeError(
        "MODFLOW 6 did not clearly report a successful run. "
        "Check run_stdout_flopy.txt in the baseline run folder."
    )

print("MODFLOW 6 baseline run completed successfully.")

MODFLOW 6 baseline run completed successfully.


## 11. List generated output files

This compares files before and after the run, then lists likely output files.


In [41]:
all_run_files = sorted([p for p in BASELINE_RUN_DIR.iterdir() if p.is_file()])

output_file_table = pd.DataFrame(
    [
        {
            "file": p.name,
            "suffix": p.suffix,
            "size_MB": round(p.stat().st_size / 1_000_000, 3),
            "modified_time": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
        }
        for p in all_run_files
    ]
).sort_values(["suffix", "file"])

output_file_table


,file,suffix,size_MB,modified_time
5,lpr_ss_gwf.cbb,.cbb,209.844,2026-04-19T05:00:14
6,lpr_ss_gwf.chd,.chd,0.000,2026-04-19T03:36:54
0,hoover.sfr.csv,.csv,0.000,2026-04-19T04:59:21
8,lpr_ss_gwf.dis,.dis,0.001,2026-04-19T03:36:54
19,lpr_ss_gwf.drn,.drn,0.000,2026-04-19T03:36:54
9,lpr_ss_gwf.dis.grb,.grb,109.681,2026-04-19T04:51:32
21,lpr_ss_gwf.hds,.hds,23.112,2026-04-19T05:00:14
22,lpr_ss_gwf.ic,.ic,0.000,2026-04-19T03:36:54
1,lpr_ss.ims,.ims,0.001,2026-04-19T03:36:54
26,lpr_ss_gwf.lst,.lst,0.330,2026-04-19T05:00:14


In [42]:
likely_output_suffixes = {
    ".hds",
    ".hed",
    ".cbc",
    ".bud",
    ".lst",
    ".csv",
    ".out",
}

likely_outputs = output_file_table[
    output_file_table["suffix"].str.lower().isin(likely_output_suffixes)
    | output_file_table["file"].str.lower().str.contains("budget|head|list|obs|stdout|stderr")
].copy()

likely_outputs


,file,suffix,size_MB,modified_time
0,hoover.sfr.csv,.csv,0.000,2026-04-19T04:59:21
21,lpr_ss_gwf.hds,.hds,23.112,2026-04-19T05:00:14
26,lpr_ss_gwf.lst,.lst,0.330,2026-04-19T05:00:14
41,mfsim.lst,.lst,0.340,2026-04-19T05:00:14
44,sfr_obs.obs,.obs,0.000,2026-04-19T03:36:55
43,run_stdout_flopy.txt,.txt,0.002,2026-04-19T05:00:14


## 12. Inspect the listing file, if available

The listing file is often one of the most useful first checks after a MODFLOW run.


In [43]:
listing_candidates = sorted(BASELINE_RUN_DIR.glob("*.lst"))

if not listing_candidates:
    print("No .lst listing file found in the run folder.")
else:
    listing_path = listing_candidates[0]
    print(f"Listing file found: {listing_path.relative_to(PROJECT_ROOT)}")
    print(f"Size: {listing_path.stat().st_size / 1_000_000:.3f} MB")

    listing_text = listing_path.read_text(errors="ignore")
    listing_lines = listing_text.splitlines()

    print("\n--- Last 60 lines of listing file ---")
    for line in listing_lines[-60:]:
        print(line)


Listing file found: results/mf6_runs/baseline_steady_state/lpr_ss_gwf.lst
Size: 0.330 MB

--- Last 60 lines of listing file ---

          OUT:                                     OUT:
          ----                                     ----
                 GWF =      419591.5787                   GWF =      419591.5787
            RAINFALL =           0.0000              RAINFALL =           0.0000
         EVAPORATION =           0.0000           EVAPORATION =           0.0000
              RUNOFF =           0.0000                RUNOFF =           0.0000
          EXT-INFLOW =           0.0000            EXT-INFLOW =           0.0000
         EXT-OUTFLOW =     4307960.8326           EXT-OUTFLOW =     4307960.8326
             STORAGE =           0.0000               STORAGE =           0.0000
           AUXILIARY =           0.0000             AUXILIARY =           0.0000

           TOTAL OUT =     4727552.4113             TOTAL OUT =     4727552.4113

            IN - OUT =      

## 13. Try reading head output with FloPy

Depending on the output-control settings, a binary head file may or may not be produced.

This cell looks for common head-file extensions and reads the first matching file.


In [44]:
from flopy.utils import HeadFile
import numpy as np

head_candidates = sorted(list(BASELINE_RUN_DIR.glob("*.hds")) + list(BASELINE_RUN_DIR.glob("*.hed")))

if not head_candidates:
    print("No binary head file found. This may be okay depending on the OC package settings.")
else:
    head_path = head_candidates[0]
    print(f"Head file found: {head_path.relative_to(PROJECT_ROOT)}")

    hds = HeadFile(str(head_path))
    times = hds.get_times()
    print(f"Number of output times: {len(times)}")
    print(f"Output times: {times}")



    head = hds.get_data(totim=times[-1])

    # Mask very large inactive/no-data values
    head_masked = np.where(head > 1e20, np.nan, head)

    print(f"Head array shape: {head.shape}")
    print(f"Minimum active head: {np.nanmin(head_masked):.3f}")
    print(f"Maximum active head: {np.nanmax(head_masked):.3f}")
    print(f"Mean active head:    {np.nanmean(head_masked):.3f}")


Head file found: results/mf6_runs/baseline_steady_state/lpr_ss_gwf.hds
Number of output times: 1
Output times: [np.float64(1.0)]
Head array shape: (3, 900, 1070)
Minimum active head: 939.000
Maximum active head: 1180.354
Mean active head:    1088.139


## 14. Try reading budget output with FloPy

This checks for a binary cell-budget file.


In [45]:
from flopy.utils import CellBudgetFile

budget_candidates = sorted(
    list(BASELINE_RUN_DIR.glob("*.cbb")) +
    list(BASELINE_RUN_DIR.glob("*.cbc")) +
    list(BASELINE_RUN_DIR.glob("*.bud"))
)

if not budget_candidates:
    print("No binary budget file found. This may be okay depending on the OC package settings.")
else:
    budget_path = budget_candidates[0]
    print(f"Budget file found: {budget_path.relative_to(PROJECT_ROOT)}")

    cbb = CellBudgetFile(str(budget_path), precision="double")
    records = cbb.get_unique_record_names()
    times = cbb.get_times()

    print(f"Number of budget output times: {len(times)}")
    print("Budget record names:")
    for record in records:
        print(f"  - {record}")


Budget file found: results/mf6_runs/baseline_steady_state/lpr_ss_gwf.cbb
Number of budget output times: 1
Budget record names:
  - b'    FLOW-JA-FACE'
  - b'      DATA-SPDIS'
  - b'        DATA-SAT'
  - b'             WEL'
  - b'             DRN'
  - b'            RCHA'
  - b'             CHD'
  - b'             SFR'


## 15. Save a baseline run summary table

The summary is saved to:

```text
results/mf6_runs/baseline_steady_state_run_summary.csv
```


In [46]:
summary_rows = [
    {"item": "project_root", "value": str(PROJECT_ROOT)},
    {"item": "source_model_folder", "value": str(LPR_MF6_SOURCE_DIR.relative_to(PROJECT_ROOT))},
    {"item": "baseline_run_folder", "value": str(BASELINE_RUN_DIR.relative_to(PROJECT_ROOT))},
    {"item": "python_executable", "value": sys.executable},
    {"item": "mf6_executable", "value": mf6_path},
    {"item": "flopy_success", "value": success},
    {"item": "normal_termination", "value": normal_termination},
    {"item": "run_success", "value": run_success},
    {"item": "elapsed_time", "value": str(elapsed)},
    {"item": "nlay", "value": nlay},
    {"item": "nrow", "value": nrow},
    {"item": "ncol", "value": ncol},
    {"item": "total_cells", "value": nlay * nrow * ncol},
    {"item": "run_timestamp", "value": end_time.isoformat(timespec="seconds")},
]

baseline_summary = pd.DataFrame(summary_rows)
summary_path = MF6_RESULTS_DIR / "baseline_steady_state_run_summary.csv"
baseline_summary.to_csv(summary_path, index=False)

print(f"Saved summary to: {summary_path.relative_to(PROJECT_ROOT)}")
baseline_summary


Saved summary to: results/mf6_runs/baseline_steady_state_run_summary.csv


,item,value
0,project_root,/workspaces/Modeling-Uncertainties
1,source_model_folder,models/lpr_mf6/steady_state
2,baseline_run_folder,results/mf6_runs/baseline_steady_state
3,python_executable,/opt/conda/envs/gw_uncertainty/bin/python
4,mf6_executable,/opt/conda/envs/gw_uncertainty/bin/mf6
5,flopy_success,True
6,normal_termination,True
7,run_success,True
8,elapsed_time,0:08:50.476507
9,nlay,3


## 16. Final interpretation

If this notebook ran successfully, then:

```text
models/lpr_mf6/steady_state/
```

is a valid clean source model folder, and:

```text
results/mf6_runs/baseline_steady_state/
```

is a valid runnable MODFLOW 6 workspace.

Recommended next steps:

1. Commit the executed notebook if you want GitHub to show the successful run.
2. Do **not** commit large generated binary outputs unless you intentionally want them version-controlled.
3. Start a PyCAP baseline run/check notebook next:

```text
02_pycap_baseline_run_check.ipynb
```

That next notebook will confirm that the analytical model side of the project can be rerun or inspected cleanly.
